<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula09a_%C3%A1rvore_de_decis%C3%A3o_com_atributos_categ%C3%B3ricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ucimlrepo -q

In [6]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
car_evaluation = fetch_ucirepo(id=19)

# data (as pandas dataframes)
X = car_evaluation.data.features.to_numpy()
y = car_evaluation.data.targets.to_numpy().reshape(-1,)

print(type(X), type(y))
print(X.shape, y.shape)


# metadata
print(car_evaluation.metadata)

# variable information
print(car_evaluation.variables)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(1728, 6) (1728,)
{'uci_id': 19, 'name': 'Car Evaluation', 'repository_url': 'https://archive.ics.uci.edu/dataset/19/car+evaluation', 'data_url': 'https://archive.ics.uci.edu/static/public/19/data.csv', 'abstract': 'Derived from simple hierarchical decision model, this database may be useful for testing constructive induction and structure discovery methods.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1728, 'num_features': 6, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5JP48', 'creators': ['Marko Bohanec'], 'intro_paper': {'ID': 249, 'type': 'NATIVE', 'title': 'Knowledge acquisition and explanation for multi-attribute decision making', 'authors': 'M. Bohanec, V. Rajkovič', 'ven

In [8]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
# # ERROR
# model.fit(X, y)
# y_pred = model.predict(X)

In [9]:
for i in range(X.shape[1]):
  values = set(X[:,i])
  print(f"{i}. {car_evaluation.variables['name'][i]}:\t{values}")

0. buying:	{'low', 'high', 'med', 'vhigh'}
1. maint:	{'low', 'high', 'med', 'vhigh'}
2. doors:	{'3', '4', '2', '5more'}
3. persons:	{'4', '2', 'more'}
4. lug_boot:	{'big', 'small', 'med'}
5. safety:	{'high', 'med', 'low'}


In [10]:
print(set(y))

{'good', 'acc', 'unacc', 'vgood'}


In [27]:
import numpy as np
labels, counts = np.unique(y, return_counts=True)
for i in range(len(labels)):
  print(f"{labels[i]}:\t{counts[i]}\t({100*counts[i]/len(y):.2f}%)")

acc:	384	(22.22%)
good:	69	(3.99%)
unacc:	1210	(70.02%)
vgood:	65	(3.76%)


In [28]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

def most_common(a):
  values, counts = np.unique(a, return_counts=True)
  return values[np.argmax(counts)]

class ZeroR(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.answer = most_common(y)
  def predict(self, X):
    return np.full(X.shape[0], self.answer)

model = ZeroR()
model.fit(X, y)
y_pred = model.predict(X)
print(f"Accuracy: {accuracy_score(y, y_pred)}")

Accuracy: 0.7002314814814815


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y,
                                                    shuffle=True,
                                                    random_state=42)
model = ZeroR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.6994219653179191


In [41]:
feature = np.random.randint(X.shape[1])
value = np.random.choice(X[:,feature])
print(f"Feature: {feature}\nValue: {value}")

Feature: 0
Value: med


In [65]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.feature = np.random.randint(X.shape[1])
    self.value = np.random.choice(X[:,self.feature])
    equals = X[:,self.feature] == self.value
    if sum(equals) == 0 or sum(~equals) == 0:
      self.answer = most_common(y)
    else:
      self.equals = DecisionTree()
      self.equals.fit(X[equals], y[equals])
      self.not_equals = DecisionTree()
      self.not_equals.fit(X[~equals], y[~equals])
  def predict(self, X):
    if hasattr(self, 'answer'):
      return np.full(X.shape[0], self.answer)
    equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.7283236994219653


In [75]:
labels, counts = np.unique(y, return_counts=True)
p = counts/sum(counts)
for i in range(len(labels)):
  print(f"{labels[i]}:\t{counts[i]}\t({p[i]:.3f})\tp^2={p[i]**2:.5f}")

print(f"Gini: {1-sum(p**2)}")

acc:	384	(0.222)	p^2=0.04938
good:	69	(0.040)	p^2=0.00159
unacc:	1210	(0.700)	p^2=0.49032
vgood:	65	(0.038)	p^2=0.00141
Gini: 0.457283763074417


In [76]:
def gini(y):
  labels, counts = np.unique(y, return_counts=True)
  p = counts/sum(counts)
  return 1-sum(p**2)

print(gini(y))

0.457283763074417


In [77]:
print(gini(np.ones(100)))

0.0


In [78]:
print(gini(np.arange(100)))

0.99


In [82]:
def impurity_value(x, y, value, impurity_function):
  equals = x == value
  equals_impurity = impurity_function(y[equals])
  not_equals_impurity = impurity_function(y[~equals])
  return (equals_impurity*sum(equals) + not_equals_impurity*sum(~equals))/len(y)

print(impurity_value(X[:,0], y, 'high', gini))

0.4551977952103337


In [83]:
def impurity_split(x, y, impurity_function):
  best_value = None
  best_impurity = np.inf
  for value in set(x):
    impurity = impurity_value(x, y, value, impurity_function)
    if impurity < best_impurity:
      best_impurity = impurity
      best_value = value
  return best_value, best_impurity

print(impurity_split(X[:,0], y, gini))

('vhigh', np.float64(0.44934645776177407))


In [85]:
def best_feature(X, y, impurity_function):
  best_feature = None
  best_value = None
  best_impurity = np.inf
  for feature in range(X.shape[1]):
    value, impurity = impurity_split(X[:,feature], y, impurity_function)
    if impurity < best_impurity:
      best_impurity = impurity
      best_feature = feature
      best_value = value
  return best_feature, best_value, best_impurity

print(best_feature(X, y, gini))

(3, '2', np.float64(0.3861571260931071))


In [86]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) == 0 or sum(~equals) == 0:
      self.answer = most_common(y)
    else:
      self.equals = DecisionTree()
      self.equals.fit(X[equals], y[equals])
      self.not_equals = DecisionTree()
      self.not_equals.fit(X[~equals], y[~equals])
  def predict(self, X):
    if hasattr(self, 'answer'):
      return np.full(X.shape[0], self.answer)
    equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9682080924855492


In [87]:
model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_train)
print(f"Accuracy: {accuracy_score(y_train, y_pred)}")

Accuracy: 1.0


In [88]:
from sklearn.model_selection import cross_val_score

model = DecisionTree()
scores = cross_val_score(model, X, y, cv=5)
print(f"Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

Accuracy: 0.761 +/- 0.085


In [92]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def __init__(self, min_sample_split=1):
    self.min_sample_split = min_sample_split
  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) < self.min_sample_split or sum(~equals) < self.min_sample_split:
      self.answer = most_common(y)
    else:
      self.equals = DecisionTree(self.min_sample_split)
      self.equals.fit(X[equals], y[equals])
      self.not_equals = DecisionTree(self.min_sample_split)
      self.not_equals.fit(X[~equals], y[~equals])
  def predict(self, X):
    if hasattr(self, 'answer'):
      return np.full(X.shape[0], self.answer)
    equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9682080924855492


In [93]:
model = DecisionTree(10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9508670520231214


In [94]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def __init__(self, min_sample_split=1, max_depth=100):
    self.min_sample_split = min_sample_split
    self.max_depth = max_depth
  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(X, y, gini)
    equals = X[:,self.feature] == self.value
    if sum(equals) < self.min_sample_split or sum(~equals) < self.min_sample_split or self.max_depth == 0:
      self.answer = most_common(y)
    else:
      self.equals = DecisionTree(self.min_sample_split, self.max_depth-1)
      self.equals.fit(X[equals], y[equals])
      self.not_equals = DecisionTree(self.min_sample_split, self.max_depth-1)
      self.not_equals.fit(X[~equals], y[~equals])
  def predict(self, X):
    if hasattr(self, 'answer'):
      return np.full(X.shape[0], self.answer)
    equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9682080924855492


In [96]:
model = DecisionTree(1, 10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9595375722543352


In [97]:
!pip install optuna -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.7 MB/s eta 0:00:00


In [98]:
import optuna
from sklearn.model_selection import KFold

def objective(trial):
  min_sample_split = trial.suggest_int('min_sample_split', 1, 20)
  max_depth = trial.suggest_int('max_depth', 1, 100)
  model = DecisionTree(min_sample_split, max_depth)
  scores = cross_val_score(model, X_train, y_train, cv=KFold(shuffle=True))
  return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)

print("Evaluation")
model = DecisionTree(**study.best_params)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

[I 2026-09-15 15:52:44,119] A new study created in memory with name: no-name-0079b4c0-5ad6-4476-a468-6ad90c8a760a
[I 2026-09-15 15:52:44,468] Trial 0 finished with value: 0.8553419138806049 and parameters: {'min_sample_split': 7, 'max_depth': 5}. Best is trial 0 with value: 0.8553419138806049.
[I 2026-09-15 15:52:45,111] Trial 1 finished with value: 0.9435436613823052 and parameters: {'min_sample_split': 3, 'max_depth': 73}. Best is trial 1 with value: 0.9435436613823052.
[I 2026-09-15 15:52:45,720] Trial 2 finished with value: 0.9370350023544184 and parameters: {'min_sample_split': 4, 'max_depth': 70}. Best is trial 1 with value: 0.9435436613823052.
[I 2026-09-15 15:52:46,131] Trial 3 finished with value: 0.8972714906084864 and parameters: {'min_sample_split': 15, 'max_depth': 90}. Best is trial 1 with value: 0.9435436613823052.
[I 2026-09-15 15:52:46,836] Trial 4 finished with value: 0.9595013864908701 and parameters: {'min_sample_split': 2, 'max_depth': 95}. Best is trial 4 with val

{'min_sample_split': 1, 'max_depth': 23}
0.9696018416784387
Evaluation
Accuracy: 0.9682080924855492
